# 01 – Load raw data & first overview

**Purpose:** load the transaction CSV and the XLS, check structure and data quality,
and save the transactions as Parquet so later notebooks load in seconds.

**Output to share:** `outputs/tables/01_overview.txt` (a text summary of everything below).

In [1]:
import sys, csv
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))  # make src/ importable from notebooks/
from src.config import DATA_RAW, DATA_PROCESSED, TABLES, TRANSACTIONS_PARQUET, ENTRY_TYPES, DEMAND_TYPES

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Collect everything important in one summary file
summary = []
def log(title, obj=""):
    text = obj.to_string() if isinstance(obj, (pd.DataFrame, pd.Series)) else str(obj)
    block = f"\n=== {title} ===\n{text}"
    print(block)
    summary.append(block)

csv_files = sorted(DATA_RAW.glob("*.csv"))
xls_files = sorted(list(DATA_RAW.glob("*.xls")) + list(DATA_RAW.glob("*.xlsx")))
print("CSV:", [f.name for f in csv_files])
print("XLS:", [f.name for f in xls_files])
assert csv_files, "Put the CSV file into data/raw/"
CSV_PATH = csv_files[0]

CSV: ['anonymized_202602.csv']
XLS: ['df_anonymized_sample.xlsx']


## 1. Peek at the raw file (encoding, delimiter, header)

In [2]:
with open(CSV_PATH, "rb") as f:
    raw_head = f.read(50_000)
raw_head = raw_head[: raw_head.rfind(b"\n")]  # cut at a full line

for ENCODING in ("utf-8-sig", "latin-1"):
    try:
        sample = raw_head.decode(ENCODING)
        break
    except UnicodeDecodeError:
        continue

SEP = csv.Sniffer().sniff("\n".join(sample.splitlines()[:50]), delimiters=",;\t|").delimiter
log("File", f"{CSV_PATH.name} | {CSV_PATH.stat().st_size / 1e6:.0f} MB | encoding={ENCODING} | sep={SEP!r}")
log("First 5 raw lines", "\n".join(sample.splitlines()[:5]))


=== File ===
anonymized_202602.csv | 632 MB | encoding=utf-8-sig | sep=','

=== First 5 raw lines ===
article_nr,date,inventory_level,quantity,entry_type,last_direct_cost,safety_stock,vendor_nr,moq,lead_time,eshop,base_unit,submarket,item_main_group,item_sub_group,item_category,product_category
1427686,2022-01-04,7.775246617380839,-2.2214990335373823,5,0.6763842681183506,0,PEDW62518913165974,0,0,False,Base Unit of Measure 1,Submarket 1,Item Main Group Code 1,Item Sub Group Code 1,Item Category Code 1,Product Group Code 1
1427686,2021-12-24,0.0,7.775246617380839,2,0.6763842681183506,0,PEDW62518913165974,0,0,False,Base Unit of Measure 1,Submarket 1,Item Main Group Code 1,Item Sub Group Code 1,Item Category Code 1,Product Group Code 1
1427676,2022-01-19,1.1107495167686912,-1.1107495167686912,5,0.5411074144946805,0,PEDW62518913165974,0,0,False,Base Unit of Measure 1,Submarket 1,Item Main Group Code 1,Item Sub Group Code 1,Item Category Code 1,Product Group Code 1
1427676,2021-12-23,0.0,1.

## 2. Load the CSV

In [3]:
header = pd.read_csv(CSV_PATH, sep=SEP, encoding=ENCODING, nrows=0).columns.str.strip()
id_cols = ["article_nr", "vendor_nr", "base_unit", "submarket", "item_category",
           "product_group", "item_main_group", "item_sub_group"]
dtypes = {c: str for c in id_cols if c in header}  # keep IDs as text (no lost leading zeros)

df = pd.read_csv(CSV_PATH, sep=SEP, encoding=ENCODING, dtype=dtypes, low_memory=False)
df.columns = df.columns.str.strip()

log("Shape (rows, columns)", df.shape)
log("Memory (MB)", round(df.memory_usage(deep=True).sum() / 1e6))
log("Columns & dtypes", df.dtypes)
df.head()


=== Shape (rows, columns) ===
(2694849, 17)

=== Memory (MB) ===
760

=== Columns & dtypes ===
article_nr              str
date                    str
inventory_level     float64
quantity            float64
entry_type            int64
last_direct_cost    float64
safety_stock          int64
vendor_nr               str
moq                   int64
lead_time             int64
eshop                  bool
base_unit               str
submarket               str
item_main_group         str
item_sub_group          str
item_category           str
product_category        str


,article_nr,date,inventory_level,quantity,entry_type,last_direct_cost,safety_stock,vendor_nr,moq,lead_time,eshop,base_unit,submarket,item_main_group,item_sub_group,item_category,product_category
0,1427686,2022-01-04,7.775247,-2.221499,5,0.676384,0,PEDW62518913165974,0,0,False,Base Unit of Measure 1,Submarket 1,Item Main Group Code 1,Item Sub Group Code 1,Item Category Code 1,Product Group Code 1
1,1427686,2021-12-24,0.000000,7.775247,2,0.676384,0,PEDW62518913165974,0,0,False,Base Unit of Measure 1,Submarket 1,Item Main Group Code 1,Item Sub Group Code 1,Item Category Code 1,Product Group Code 1
2,1427676,2022-01-19,1.110750,-1.110750,5,0.541107,0,PEDW62518913165974,0,0,False,Base Unit of Measure 1,Submarket 1,Item Main Group Code 1,Item Sub Group Code 1,Item Category Code 1,Product Group Code 1
3,1427676,2021-12-23,0.000000,1.110750,2,0.541107,0,PEDW62518913165974,0,0,False,Base Unit of Measure 1,Submarket 1,Item Main Group Code 1,Item Sub Group Code 1,Item Category Code 1,Product Group Code 1
4,1427675,2022-01-12,4.442998,-4.442998,5,173.993089,0,BEGM32354199798630,0,0,False,Base Unit of Measure 1,Submarket 1,Item Main Group Code 2,Item Sub Group Code 2,Item Category Code 2,Product Group Code 2


## 3. Missing values

In [4]:
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(),
})
log("Missing values & cardinality per column", missing)


=== Missing values & cardinality per column ===
                  missing  missing_%  n_unique
article_nr              0        0.0     58764
date                    0        0.0      3731
inventory_level         0        0.0    150635
quantity                0        0.0      9544
entry_type              0        0.0         7
last_direct_cost        0        0.0     37152
safety_stock            0        0.0       108
vendor_nr               0        0.0       246
moq                     0        0.0       109
lead_time               0        0.0        26
eshop                   0        0.0         2
base_unit               0        0.0        11
submarket               0        0.0        18
item_main_group         0        0.0        11
item_sub_group          0        0.0        43
item_category           0        0.0        43
product_category        0        0.0       340


## 4. Date column
Check the raw format first – day/month order matters (Swiss data is usually day-first).

In [5]:
log("Sample raw date values", df["date"].dropna().astype(str).sample(10, random_state=0).tolist())

df["date"] = pd.to_datetime(df["date"], errors="coerce", format="mixed", dayfirst=True)
log("Date range", f"{df['date'].min()}  ->  {df['date'].max()}  | unparsed: {df['date'].isna().sum()}")
log("Transactions per year", df["date"].dt.year.value_counts().sort_index())


=== Sample raw date values ===
['2016-09-30', '2021-11-30', '2019-10-18', '2020-02-19', '2009-04-01', '2018-09-20', '2016-09-27', '2009-04-17', '2009-01-28', '2019-11-21']

=== Date range ===
2007-01-10 00:00:00  ->  2022-12-01 00:00:00  | unparsed: 0

=== Transactions per year ===
date
2007     53566
2008    108314
2009    105468
2010    127653
2011    133368
2012    153235
2013    168448
2014    163358
2015    186121
2016    215157
2017    234641
2018    245771
2019    257347
2020    255747
2021    274127
2022     12528


## 5. Entry types
Also shows the sign of `quantity` per type – we need to know if outflows (sales, usage) are negative.

In [6]:
et = (df.groupby("entry_type")["quantity"]
        .agg(count="size", min="min", median="median", max="max", sum="sum")
        .round(2))
et.insert(0, "label", et.index.map(ENTRY_TYPES))
log("Entry types (with quantity stats)", et)


=== Entry types (with quantity stats) ===
                              label    count        min  median        max           sum
entry_type                                                                              
0                       Procurement   133443 -191048.92   11.11  737160.02  1.396552e+08
1                              Sale   432731  -66644.97   -4.44   24991.86 -5.077687e+07
2                Ingoing adjustment    18615       0.00   13.33  251054.94  6.976834e+06
3               Outgoing adjustment    26598 -251029.39   -7.78       2.22 -5.353015e+06
4              Internal transaction   115275 -493304.96    1.11  370897.04 -1.165180e+03
5              Usage for production  1748717  -40931.12   -8.89     566.48 -8.921864e+07
6           Ingoing from production   219470     -33.32    3.33   40931.12  5.372881e+06


## 6. Articles & the inventory-article population
`safety_stock` is stored per transaction, so we use each article's **latest** value
to decide whether it is an inventory article today (definition: `safety_stock > 0`).

In [7]:
latest = df.sort_values("date").groupby("article_nr").tail(1).set_index("article_nr")
ever_ss = df.groupby("article_nr")["safety_stock"].max() > 0

inv_articles = latest.index[latest["safety_stock"] > 0]
log("Article counts", pd.Series({
    "all articles": df["article_nr"].nunique(),
    "inventory articles (latest safety_stock > 0)": len(inv_articles),
    "articles with safety_stock > 0 at any point": int(ever_ss.sum()),
}))

inv = latest.loc[inv_articles]
log("Inventory articles – lead_time", pd.Series({
    "lead_time == 0 (not maintained)": int((inv["lead_time"] == 0).sum()),
    "lead_time missing": int(inv["lead_time"].isna().sum()),
    "lead_time > 0": int((inv["lead_time"] > 0).sum()),
}))
log("Inventory articles – describe numeric (latest values)",
    inv[["lead_time", "safety_stock", "moq", "last_direct_cost", "inventory_level"]].describe().round(2))
log("Inventory articles – base_unit (top 15)", inv["base_unit"].value_counts().head(15))


=== Article counts ===
all articles                                    58764
inventory articles (latest safety_stock > 0)     1751
articles with safety_stock > 0 at any point      1751

=== Inventory articles – lead_time ===
lead_time == 0 (not maintained)      37
lead_time missing                     0
lead_time > 0                      1714

=== Inventory articles – describe numeric (latest values) ===
       lead_time  safety_stock        moq  last_direct_cost  inventory_level
count    1751.00       1751.00    1751.00           1751.00          1751.00
mean       26.27       1036.05     759.28             19.75          2722.70
std        20.69       4203.19    4467.59             48.98         11312.04
min         0.00          1.00       0.00              0.00           -19.99
25%        15.00         11.00       0.00              0.58            23.88
50%        30.00         55.00      44.00              4.63           123.29
75%        30.00        277.00     222.00           

## 7. Sanity check against the PDF: last time an inventory article was sold/used
The PDF chart shows 1966 / 818 / 400 / 239 / 133 / 60. If we get similar numbers,
our definitions (inventory article, demand = entry types 1 & 5) match theirs.

In [8]:
ref_date = df["date"].max()
last_use = (df[df["entry_type"].isin(DEMAND_TYPES) & df["article_nr"].isin(inv_articles)]
              .groupby("article_nr")["date"].max())
years_since = (ref_date - last_use).dt.days / 365.25

bins = [-np.inf, 1, 2, 3, 4, 5, np.inf]
labels = ["<1 year", ">1 year", ">2 years", ">3 years", ">4 years", ">5 years"]
last_use_dist = pd.cut(years_since, bins=bins, labels=labels).value_counts().reindex(labels)
log(f"Last sale/usage of inventory articles (reference date {ref_date.date()})", last_use_dist)
log("Inventory articles with no sale/usage at all", len(inv_articles) - len(last_use))


=== Last sale/usage of inventory articles (reference date 2022-12-01) ===
date
<1 year     1060
>1 year      380
>2 years      50
>3 years      11
>4 years       4
>5 years      32

=== Inventory articles with no sale/usage at all ===
214


## 8. Save as Parquet

In [9]:
for c in [c for c in df.columns if df[c].dtype == object]:
    df[c] = df[c].astype("string")  # Parquet needs consistent types

df.to_parquet(TRANSACTIONS_PARQUET, index=False)
log("Saved", f"{TRANSACTIONS_PARQUET.name} | {TRANSACTIONS_PARQUET.stat().st_size / 1e6:.0f} MB")


=== Saved ===
transactions.parquet | 15 MB


## 9. The XLS file – what's in it?

In [10]:
for path in xls_files:
    xl = pd.ExcelFile(path)
    log(f"XLS {path.name} – sheets", xl.sheet_names)
    for sheet in xl.sheet_names:
        s = xl.parse(sheet)
        log(f"Sheet '{sheet}' – shape {s.shape}, columns", list(s.columns))
        log(f"Sheet '{sheet}' – first rows", s.head(8))


=== XLS df_anonymized_sample.xlsx – sheets ===
['Sheet1']

=== Sheet 'Sheet1' – shape (10, 17), columns ===
['article_nr', 'date', 'inventory_level', 'quantity', 'entry_type', 'last_direct_cost', 'safety_stock', 'vendor_nr', 'moq', 'lead_time', 'eshop', 'base_unit', 'submarket', 'item_main_group', 'item_sub_group', 'item_category', 'product_group']

=== Sheet 'Sheet1' – first rows ===
  article_nr       date  inventory_level  quantity  entry_type  last_direct_cost  safety_stock     vendor_nr  moq  lead_time  eshop               base_unit    submarket         item_main_group         item_sub_group         item_category         product_group
0  Art.No. 1 2021-12-24         0.000000  7.775247           2          0.676384             0  Vendor No. 1    0          0  False  Base Unit of Measure 1  Submarket 1  Item Main Group Code 1  Item Sub Group Code 1  Item Category Code 1  Product Group Code 1
1  Art.No. 1 2022-01-04         7.775247 -2.221499           5          0.676384           

## 10. Write summary file

In [11]:
out = TABLES / "01_overview.txt"
out.write_text("\n".join(summary), encoding="utf-8")
print(f"Summary written to {out} – share this file.")

Summary written to /Users/nikolaosgeorgantopoulos/Repositories/inventory-optimization/outputs/tables/01_overview.txt – share this file.
